# Week 8, Day 6: Forge: Week 8 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
f = lambda x: x ** 3 * np.exp(x)
print("1.", (f(1 + 1e-6) - f(1 - 1e-6)) / 2e-6, " exact 4e =", 4 * np.e)
print("2.", 1 - np.tanh(0.0) ** 2)
print("3.", (2 * 1 * 3, 1 ** 2))

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(2020)
N, T = 50, 800
prefs = np.linspace(0, 180, N, endpoint=False)
def wrap(d): return (d + 90) % 180 - 90
def tune(theta): return 5 + 40 * np.exp(-wrap(theta[:, None] - prefs[None]) ** 2 / (2 * 25 ** 2))
delta = rng.uniform(-4, 4, T)                                     # offset from 45 degrees
labels = (delta > 0).astype(float)                                # 1 = clockwise
counts = rng.poisson(tune(45 + delta) * 0.5)                      # 0.5 s spike counts, (800, 50)
train, test = np.arange(600), np.arange(600, 800)
mu, sd = counts[train].mean(axis=0), counts[train].std(axis=0) + 1e-9
Zb = np.column_stack([(counts - mu) / sd, np.ones(T)])           # z-scored responses plus a bias column
def sigmoid(u): return 1 / (1 + np.exp(-u))
print(Zb.shape)

### Exercise: Part A: derive, check and train a logistic decoder

Write `xent(w, Z, y)` and `grad_xent(w, Z, y)`. The gradient has a famously simple form, $\nabla L = \frac{1}{T}Z^\top(\mathbf{p} - \mathbf{y})$, which you prove in the derivation set. Check it numerically (the test does this too), then train for 1000 steps with learning rate 0.5 on the training trials and report the accuracy on the test trials, `acc_test`.

In [ ]:
def xent(w, Z, y):
    p = sigmoid(Z @ w)
    return ...

def grad_xent(w, Z, y):
    return ...

w = np.zeros(N + 1)
for step in range(1000):
    ...
acc_test = np.mean((sigmoid(Zb[test] @ w) > 0.5) == labels[test])

order = np.argsort(prefs)
plt.figure(figsize=(5, 3)); plt.plot(prefs[order], w[:N][order], "o-", ms=3); plt.axvline(45, ls="--", color="0.5")
plt.xlabel("neuron's preferred orientation (deg)"); plt.ylabel("decoder weight"); plt.show()
print("test accuracy", round(acc_test, 3))

In [ ]:
import numpy as np
def test_loss():
    "xent computes the mean cross-entropy"
    w = np.zeros(N + 1)
    assert np.isclose(xent(w, Zb[train], labels[train]), np.log(2))

def test_grad():
    "grad_xent matches a numerical gradient"
    w = np.random.default_rng(0).normal(size=N + 1) * 0.1
    num = np.zeros(N + 1)
    for i in range(N + 1):
        e = np.zeros(N + 1); e[i] = 1e-6
        num[i] = (xent(w + e, Zb[train], labels[train]) - xent(w - e, Zb[train], labels[train])) / 2e-6
    assert np.allclose(grad_xent(w, Zb[train], labels[train]), num, atol=1e-6)

def test_accuracy():
    "after training, test accuracy is above 75%"
    assert acc_test > 0.75

test_loss()
test_grad()
test_accuracy()
print("All checks passed")

### Exercise: Part B: gradients through deep stacks

A 10-layer network of width 50 has random weights with standard deviation scale/√50. Complete `layer_grad_norms`, which sends a random error back from the top and records the norm of the gradient with respect to each layer's pre-activations, from layer 1 to 10. Compute the ratio of layer 1's norm to layer 10's for sigmoid and tanh (scale 1) and ReLU (scale √2, 'He initialisation').

In [ ]:
def layer_grad_norms(act, dact, scale, L=10, n=50):
    r = np.random.default_rng(1)
    W = [r.normal(size=(n, n)) * scale / np.sqrt(n) for _ in range(L)]
    h = r.normal(size=(64, n)); zs = []
    for l in range(L):                       # forward pass, keeping pre-activations
        z = h @ W[l]; zs.append(z); h = act(z)
    g = r.normal(size=h.shape)               # an error arriving at the top layer's output
    norms = []
    for l in reversed(range(L)):
        gz = ...                             # through the nonlinearity of layer l
        norms.append(np.linalg.norm(gz))
        g = ...                              # back through layer l's weights to its input
    return np.array(norms[::-1])

acts = {"sigmoid": (sigmoid, lambda z: sigmoid(z) * (1 - sigmoid(z)), 1.0),
        "tanh": (np.tanh, lambda z: 1 - np.tanh(z) ** 2, 1.0),
        "relu": (lambda z: np.maximum(z, 0), lambda z: (z > 0) * 1.0, np.sqrt(2))}
ratios = {}
plt.figure(figsize=(5, 3.2))
for name, (a, da, sc) in acts.items():
    nr = layer_grad_norms(a, da, sc)
    ratios[name] = nr[0] / nr[-1]
    plt.semilogy(range(1, 11), nr, "o-", label=name)
plt.xlabel("layer"); plt.ylabel("gradient norm"); plt.legend(frameon=False); plt.show()
print({k: f"{v:.2e}" for k, v in ratios.items()})

In [ ]:
import numpy as np
def test_norms():
    "layer_grad_norms returns one norm per layer"
    out = layer_grad_norms(np.tanh, lambda z: 1 - np.tanh(z) ** 2, 1.0, L=6)
    assert out.shape == (6,)

def test_ratios():
    "sigmoid gradients collapse, tanh shrink moderately, ReLU with He scaling stay roughly constant"
    assert ratios["sigmoid"] < 1e-4
    assert 0.01 < ratios["tanh"] < 1
    assert 0.3 < ratios["relu"] < 3

test_norms()
test_ratios()
print("All checks passed")

In [ ]:
rng2 = np.random.default_rng(1986)
theta = rng2.uniform(-np.pi, np.pi, 200)
X = np.column_stack([np.cos(theta), np.sin(theta)])
y = 10 + 30 * np.exp(2 * (np.cos(theta - 0.8) - 1)) + 2 * rng2.normal(size=200)
init = {"W1": rng2.normal(size=(2, 16)) * 0.8, "b1": np.zeros(16), "w2": rng2.normal(size=16) * 0.5, "b2": 0.0}
B = rng2.normal(size=16) * 0.5                     # fixed random feedback weights
def forward(p, X):
    H = np.tanh(X @ p["W1"] + p["b1"])
    return H, H @ p["w2"] + p["b2"]
def mse(p): return 0.5 * np.mean((forward(p, X)[1] - y) ** 2)

### Exercise: Part C: learning with random feedback

Complete `train(feedback)`: it is Day 5's gradient descent, except that the error sent to the hidden layer is `np.outer(d_yhat, fb)`, where `fb` is `p['w2']` for backpropagation and the fixed `B` for feedback alignment. Train both for 3000 steps and compare the loss curves. Finally measure the cosine between the learned `w2` and `B`.

In [ ]:
def train(feedback, steps=3000, lr=0.05):
    p = {k: np.array(v, dtype=float, copy=True) for k, v in init.items()}
    losses = []
    for _ in range(steps):
        H, yhat = forward(p, X)
        d_yhat = (yhat - y) / len(y)
        fb = p["w2"] if feedback == "bp" else B
        dZ = ...
        p["W1"] -= lr * X.T @ dZ
        p["b1"] -= lr * dZ.sum(axis=0)
        p["w2"] -= lr * H.T @ d_yhat
        p["b2"] -= lr * d_yhat.sum()
        losses.append(mse(p))
    return p, losses

p_bp, loss_bp = train("bp")
p_fa, loss_fa = train("fa")
align_after = ...

plt.figure(figsize=(5, 3.2)); plt.semilogy(loss_bp, label="backpropagation"); plt.semilogy(loss_fa, label="feedback alignment")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(frameon=False); plt.show()
print(f"final loss: BP {loss_bp[-1]:.2f}, FA {loss_fa[-1]:.2f};  cosine(w2, B) after FA training = {align_after:.2f}")

In [ ]:
import numpy as np
def test_curves():
    "both training runs record 3000 losses"
    assert len(loss_bp) == 3000 and len(loss_fa) == 3000

def test_learns():
    "feedback alignment reduces the loss by more than a factor of 10, as backpropagation does"
    assert loss_bp[-1] < loss_bp[0] / 10
    assert loss_fa[-1] < loss_fa[0] / 10

def test_alignment():
    "after training, the forward weights w2 have come to align with B (positive cosine)"
    assert align_after > 0.3

test_curves()
test_learns()
test_alignment()
print("All checks passed")